# Experiment 10: Unsupervised & Pattern Discovery Analysis

This notebook analyzes results from Experiment 10, which addresses supervisor feedback on unsupervised methods:
- **10A**: Anomaly Detection (Isolation Forest, One-Class SVM, LOF)
- **10B**: Clustering (K-Means, DBSCAN) for fraud ring discovery
- **10C**: Association Rules for interpretable fraud patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
ARTIFACTS_DIR = Path('../artifacts/unsupervised')

print(f"Loading results from: {ARTIFACTS_DIR}")
print(f"Available files: {list(ARTIFACTS_DIR.glob('*.csv'))}")

## 10A: Anomaly Detection Comparison

**Research Question**: Can unsupervised anomaly detection match supervised XGBoost?

**Methods tested**:
- Isolation Forest (HIGH priority)
- One-Class SVM (HIGH priority)
- Local Outlier Factor (MEDIUM priority)

In [ ]:
# Load anomaly detection results
anomaly_results = pd.read_csv(ARTIFACTS_DIR / 'anomaly_detection_results.csv')
display(anomaly_results)

# Calculate improvement
supervised_auc = anomaly_results[anomaly_results['type'] == 'supervised']['auc_pr'].iloc[0]
best_unsupervised_auc = anomaly_results[anomaly_results['type'] == 'unsupervised']['auc_pr'].max()
improvement = supervised_auc / best_unsupervised_auc

print(f"\n🔑 Key Finding:")
print(f"   Supervised XGBoost: {supervised_auc:.3f} AUC-PR")
print(f"   Best Unsupervised:  {best_unsupervised_auc:.3f} AUC-PR")
print(f"   Improvement:        {improvement:.1f}x better")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {'unsupervised': '#3498db', 'supervised': '#e74c3c'}
bar_colors = [colors[t] for t in anomaly_results['type']]

# AUC-PR
ax1 = axes[0]
bars1 = ax1.barh(anomaly_results['method'], anomaly_results['auc_pr'], color=bar_colors)
ax1.set_xlabel('AUC-PR')
ax1.set_title('AUC-PR: Supervised vs Unsupervised')
ax1.set_xlim(0, 1)
for bar, val in zip(bars1, anomaly_results['auc_pr']):
    ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

# P@100
ax2 = axes[1]
bars2 = ax2.barh(anomaly_results['method'], anomaly_results['p_at_100'], color=bar_colors)
ax2.set_xlabel('Precision@100')
ax2.set_title('P@100: Supervised vs Unsupervised')
ax2.set_xlim(0, 1)
for bar, val in zip(bars2, anomaly_results['p_at_100']):
    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
fig.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'exp10a_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10B: Clustering Analysis (Fraud Ring Discovery)

**Research Question**: Can we identify fraud "rings" or behaviorally similar groups?

**Methods tested**:
- K-Means (k=10, 20, 50)
- DBSCAN

In [ ]:
# Load clustering results
clustering_summary = pd.read_csv(ARTIFACTS_DIR / 'clustering_summary.csv')
display(clustering_summary)

# Key metrics
best_method = clustering_summary.loc[clustering_summary['max_fraud_lift'].idxmax()]
print(f"\n🔑 Key Finding:")
print(f"   Best method: {best_method['method']}")
print(f"   High-fraud clusters: {best_method['n_high_fraud_clusters']:.0f}")
print(f"   Max fraud lift: {best_method['max_fraud_lift']:.1f}x")

In [ ]:
# Load high-fraud clusters
high_fraud_path = ARTIFACTS_DIR / 'high_fraud_clusters.csv'
if high_fraud_path.exists():
    high_fraud = pd.read_csv(high_fraud_path)
    print(f"\n📊 High-Fraud Clusters (lift > 2x):")
    display(high_fraud[['method', 'cluster', 'n_samples', 'fraud_rate', 'fraud_lift']].head(15))
else:
    print("No high-fraud clusters file found")

In [ ]:
# Load detailed cluster stats
cluster_stats = pd.read_csv(ARTIFACTS_DIR / 'cluster_fraud_rates.csv')

# Visualization: Fraud rate distribution by method
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, method in enumerate(cluster_stats['method'].unique()[:4]):
    ax = axes[i // 2, i % 2]
    method_data = cluster_stats[cluster_stats['method'] == method].sort_values('fraud_rate', ascending=False)
    
    colors = ['#e74c3c' if r > 0.16 else '#f39c12' if r > 0.08 else '#3498db' 
              for r in method_data['fraud_rate']]
    
    ax.barh(range(len(method_data)), method_data['fraud_rate'], color=colors)
    ax.axvline(x=0.082, color='red', linestyle='--', alpha=0.5, label='Overall avg (8.2%)')
    ax.set_xlabel('Fraud Rate')
    ax.set_ylabel('Cluster')
    ax.set_title(f'{method}')
    ax.legend()

plt.suptitle('Fraud Rate by Cluster', fontsize=14)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'exp10b_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10C: Association Rules (Interpretable Patterns)

**Research Question**: What interpretable rules predict fraud?

**Method**: FP-Growth association rule mining

In [ ]:
# Load top fraud rules
top_rules = pd.read_csv(ARTIFACTS_DIR / 'top_fraud_rules.csv')
print("🔑 Top Fraud-Predicting Rules:")
print("(Read as: IF <rule> THEN FRAUD)\n")
display(top_rules.head(10))

In [ ]:
# Visualization: Top rules by lift
fig, ax = plt.subplots(figsize=(12, 8))

top_10 = top_rules.head(10)
# Clean up lift values (remove 'x' suffix if present)
if top_10['Lift'].dtype == 'object':
    top_10['Lift_num'] = top_10['Lift'].str.replace('x', '').astype(float)
else:
    top_10['Lift_num'] = top_10['Lift']

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_10)))
bars = ax.barh(range(len(top_10)), top_10['Lift_num'], color=colors)
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels([r[:50] + '...' if len(r) > 50 else r for r in top_10['Rule (IF)']])
ax.set_xlabel('Fraud Lift (x times more likely)')
ax.set_title('Top 10 Fraud-Predicting Association Rules')
ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='Baseline (1x)')

for bar, val in zip(bars, top_10['Lift_num']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}x', va='center')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'exp10c_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Table for Thesis

In [ ]:
# Create summary table
summary_data = [
    {
        'Experiment': '10A: Anomaly Detection',
        'Best Method': 'XGBoost (supervised)',
        'Key Metric': f'AUC-PR: {supervised_auc:.3f}',
        'Finding': f'Supervised outperforms unsupervised by {improvement:.1f}x'
    },
    {
        'Experiment': '10B: Clustering',
        'Best Method': best_method['method'],
        'Key Metric': f"{best_method['n_high_fraud_clusters']:.0f} high-fraud clusters",
        'Finding': f"Max {best_method['max_fraud_lift']:.1f}x fraud lift"
    },
    {
        'Experiment': '10C: Association Rules',
        'Best Method': 'FP-Growth',
        'Key Metric': f"{len(top_rules)} fraud rules",
        'Finding': f"Best rule: {top_rules.iloc[0]['Lift']} lift"
    }
]

summary_df = pd.DataFrame(summary_data)
print("=" * 80)
print("EXPERIMENT 10: UNSUPERVISED & PATTERN DISCOVERY SUMMARY")
print("=" * 80)
display(summary_df)

## Key Takeaways

### 10A: Anomaly Detection
- ✅ Unsupervised methods considered (Isolation Forest, One-Class SVM, LOF)
- ❌ All underperform supervised XGBoost by **8-10x**
- 📝 **Thesis value**: Justifies supervised approach; demonstrates methodological completeness

### 10B: Clustering
- ✅ Found **high-fraud clusters** with up to **12x fraud lift**
- 📝 **Thesis value**: Fraud ring discovery for targeted investigation

### 10C: Association Rules
- ✅ `bundle_tier_premium + RENT` = **5.7x fraud lift**
- 📝 **Thesis value**: Interpretable rules for fraud analysts (SHAP complement)